# Churn — XGBoost Training

## 0. Configuration

All tuneable settings live here — change them before running the rest of the notebook.

In [ ]:
from datetime import datetime

# ── Data settings ────────────────────────────────────────────────────────
DATA_CONFIG = {
    "data_path": r"data/churn_targets.parquet",
    "target": "CHURN_2M",
    "drop_cols": [
        "CARD_ID", "CARD_CANCELLATION_DATE", "CHURN_1M", "CHURN_3M",
        "STATUS", "BIRTH_DT", "OPEN_DT", "MONTH_END", "CUST_ID",
        "LAST_EXPOSURE_DATE", "MONTH_YYYYMM",
    ],
    "random_state": 42,
    "test_month_start": datetime(2025, 2, 1),
    "month_window_start": datetime(2023, 2, 1),
    "month_window_end": datetime(2025, 1, 31),
    "majority_reduction_pct": 0.10,
    "n_lags": 12,
    "priority_non_null_cols": [
        "REMAINING_INSTALLMENT_AMOUNT_lag1",
        "REMAINING_INSTALLMENT_COUNT_lag1",
        "TOTAL_EXPOSURE_AMT_1M_lag1",
        "STMT_PYMNT_AMT_lag1",
        "AVG_TICKET_MTH_lag1",
        "MCC_MOST_FREQUENT_MTH_lag1",
    ],
    "use_streaming_collect": True,
}

# ── XGBoost hyperparameters ──────────────────────────────────────────────
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "enable_categorical": True,
    "random_state": DATA_CONFIG["random_state"],
    "n_jobs": -1,
    "eval_metric": "aucpr",
}

# ── MLflow settings ──────────────────────────────────────────────────────
EXPERIMENT = "churn-xgboost"
RUN_NAME = "xgb-baseline"

# ── Model save path (for explainability) ─────────────────────────────────
MODEL_SAVE_DIR = "training/saved_models/xgboost"

## 1. Imports & Setup

In [ ]:
import logging
import os
import sys
import time

import numpy as np
from xgboost import XGBClassifier

ROOT = os.path.abspath(".")
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "training"))

from data_processing import load_and_prepare_data
from mlflow_utils import (
    configure_training_logging,
    log_classification_results,
    log_dataset_info,
    log_feature_importance,
    save_model_local,
    start_run,
    xgboost_progress_callback,
)

configure_training_logging()
log = logging.getLogger("churn")

## 2. Load & Prepare Data

In [ ]:
X_train_pd, X_test_pd, y_train, y_test, feature_names, data_info = (
    load_and_prepare_data(DATA_CONFIG)
)

print(f"X_train: {X_train_pd.shape}  X_test: {X_test_pd.shape}")
print(f"Features: {len(feature_names)}  scale_pos_weight: {data_info['scale_pos_weight']}")

## 3. Train XGBoost

In [ ]:
mlflow_params = {
    k: v for k, v in XGB_PARAMS.items() if k != "n_jobs"
}

with start_run(EXPERIMENT, run_name=RUN_NAME, params=mlflow_params) as run:
    log_dataset_info(data_info)

    model = XGBClassifier(**XGB_PARAMS, verbosity=1)

    log.info("Starting XGBoost fit...")
    _t_fit = time.perf_counter()
    try:
        model.fit(X_train_pd, y_train, callbacks=[xgboost_progress_callback(30)])
    except TypeError:
        model.fit(X_train_pd, y_train)
    log.info("XGBoost fit finished in %.1fs", time.perf_counter() - _t_fit)

    metrics = log_classification_results(model, X_test_pd, y_test, model_flavor="xgboost")
    log_feature_importance(model, feature_names)

    run_id = run.info.run_id
    print(f"Run ID  : {run_id}")
    print(f"Metrics : {metrics}")

## 4. Feature Importance Analysis

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

imp_df = (
    pd.DataFrame({"feature": feature_names, "importance": model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 25 feature-lag columns by importance:")
print(imp_df.head(25).to_string(index=False))

imp_df["base_feature"] = imp_df["feature"].str.replace(r"_lag\d+$", "", regex=True)
base_imp = (
    imp_df.groupby("base_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 25 base features (importance summed across lags):")
print(base_imp.head(25).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_top = imp_df.head(20).iloc[::-1]
axes[0].barh(plot_top["feature"], plot_top["importance"])
axes[0].set_title("Top 20 feature-lag columns")
axes[0].set_xlabel("Importance")

plot_base = base_imp.head(20).iloc[::-1]
axes[1].barh(plot_base["base_feature"], plot_base["importance"])
axes[1].set_title("Top 20 base features (sum across lags)")
axes[1].set_xlabel("Importance")

fig.tight_layout()
plt.show()

## 5. Threshold Tuning

In [ ]:
import mlflow
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

log.info("Threshold tuning on baseline model...")

y_proba_baseline = model.predict_proba(X_test_pd)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_baseline, ax=axes[0], name="XGBoost baseline",
)
axes[0].set_title("Precision-Recall Curve")

precision, recall, thresholds = precision_recall_curve(y_test, y_proba_baseline)

precision_t = precision[:-1]
recall_t = recall[:-1]
den = precision_t + recall_t
f1s = np.where(den > 0, 2.0 * precision_t * recall_t / den, 0.0)

best_idx = int(np.nanargmax(f1s))
best_threshold = float(thresholds[best_idx])
y_pred_tuned = (y_proba_baseline >= best_threshold).astype(int)

axes[1].plot(thresholds, precision_t, label="Precision")
axes[1].plot(thresholds, recall_t, label="Recall")
axes[1].plot(thresholds, f1s, label="F1")
axes[1].axvline(
    best_threshold, color="r", linestyle="--",
    label=f"Best thr = {best_threshold:.3f} (F1={f1s[best_idx]:.3f})",
)
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("Precision / Recall / F1 vs threshold")
axes[1].legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

print(f"Best threshold (max F1): {best_threshold:.4f}")
print(f"  P={precision_t[best_idx]:.4f}  R={recall_t[best_idx]:.4f}  F1={f1s[best_idx]:.4f}")

In [ ]:
from mlflow_utils import log_hyperparameter_tuning

tuned_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_tuned),
    "precision": precision_score(y_test, y_pred_tuned, zero_division=0),
    "recall": recall_score(y_test, y_pred_tuned, zero_division=0),
    "f1": f1_score(y_test, y_pred_tuned, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_baseline),
    "avg_precision": average_precision_score(y_test, y_proba_baseline),
}

tuning_summary = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision_t,
    "recall": recall_t,
    "f1": f1s,
})

with start_run(
    EXPERIMENT,
    run_name=f"{RUN_NAME}-threshold-tuned",
    params={
        **mlflow_params,
        "threshold": round(best_threshold, 4),
        "threshold_selection_metric": "max_f1_from_pr_curve",
    },
) as run_tuned:
    mlflow.log_metrics(tuned_metrics)
    log_hyperparameter_tuning(tuning_summary, metric_col="f1")

    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_tuned, ax=ax_cm, cmap="Blues",
    )
    ax_cm.set_title(f"Confusion Matrix (threshold={best_threshold:.3f})")
    plt.show()

print(f"Tuned metrics: {tuned_metrics}")
print(f"Run ID       : {run_tuned.info.run_id}")

## 6. Compare All Runs

In [ ]:
from mlflow_utils import TRACKING_URI

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)

runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    order_by=["metrics.avg_precision DESC"],
)

key_cols = [
    "tags.mlflow.runName",
    "metrics.avg_precision", "metrics.f1",
    "metrics.precision", "metrics.recall",
    "metrics.roc_auc", "params.threshold",
]
display_cols = [c for c in key_cols if c in runs_df.columns]
comparison = runs_df[display_cols].copy()
comparison.columns = [c.rsplit(".", 1)[-1] for c in comparison.columns]

print("=== MLflow Run Comparison ===")
print(comparison.to_string(index=False))

## 7. Save Model Locally

Saves the model + full metadata so it can be loaded in the explainability notebook.

In [ ]:
model_metadata = {
    "experiment": EXPERIMENT,
    "run_name": RUN_NAME,
    "run_id": run_id,
    "model_type": "XGBClassifier",
    "hyperparameters": XGB_PARAMS,
    "feature_names": feature_names,
    "data_info": data_info,
    "best_threshold": best_threshold,
    "metrics": metrics,
    "tuned_metrics": tuned_metrics,
}

save_path = save_model_local(model, MODEL_SAVE_DIR, model_metadata)
print(f"Model saved to: {save_path}")

## 8. Cleanup

In [ ]:
import gc

log.info("Cleaning up...")
del X_train_pd, X_test_pd
gc.collect()
log.info("Done.")